In [138]:
#Paramters:

#AMOUNT OF ANIME TO LOOK AT
ANIME_AMOUNT = 2000
#The ration of Train to test taken at various points
TRAIN_TEST_SPLITS = 0.9, 0.8
#Random Seeds
SEEDS = 1564,1508,549
#Proprotinal value of users to look at.
DATA_TO_TAKE = 0.2
#Minimum amount of shows a user needs to review to be valid
MIN_SHOWS  = 50
#Maximum number of users to take account of in the reccomendation system.
KS_TO_TRY = 15


TOP_ANIME_PATH = "top_animes.csv"

#OUTPUT_FILES
OUTPUT_TRAINING_VALUES = "training_vals.npy"
OUTPUT_TRAIN_USERS = "train_users.csv"
OUTPUT_VALIDATION_USERS = "validation_users.csv"
NEW_TOP_ANIME_PATH = "new_top_animes.csv"

In [139]:
import pandas as pd
import numpy as np

from tqdm import trange

In [140]:
import kagglehub
svanoo_myanimelist_dataset_path = kagglehub.dataset_download('svanoo/myanimelist-dataset')
user = pd.read_csv(f'{svanoo_myanimelist_dataset_path}/user.csv',
                   sep = '\t', keep_default_na=False,usecols=("user_id","num_completed","mean_score")
                   ,index_col = "user_id")
top_animes = pd.read_csv(TOP_ANIME_PATH,keep_default_na=False,index_col="anime_id")
top_animes["number"] = np.arange(top_animes.shape[0])
top_animes = top_animes.head(ANIME_AMOUNT)
top_animes.to_csv(NEW_TOP_ANIME_PATH)
user["number_reviews"] = np.zeros(user.shape[0],dtype=int)

In [141]:
print(user.columns)
user.head()

Index(['num_completed', 'mean_score', 'number_reviews'], dtype='object')


,num_completed,mean_score,number_reviews
user_id,,,
kir1yama,606,6.69,0
smatster,1188,8.38,0
suzuhrevv,517,6.04,0
pheseantnetsuke,368,7.22,0
skyuchiha,1038,7.67,0


In [142]:
NUM_USER_ANIME_FILE=70
cols_to_use = ['anime_id', 'user_id', 'score', 'favorite']
for x in trange(NUM_USER_ANIME_FILE):
    file_name = "{}/user_anime{:012d}.csv".format(svanoo_myanimelist_dataset_path,x)
    df = pd.read_csv(
        file_name,
        sep='\t',
        keep_default_na=False,
        na_values=("-------",""),
        low_memory=False,
        usecols=cols_to_use  # Only read needed columns
    ).dropna()
    df = df[df["anime_id"].isin(top_animes.index)]
    df= df[df["user_id"].isin(user.index)]
    df_users=df.value_counts("user_id")
    user.loc[df_users.index,'number_reviews'] += df_users
    assert not user["number_reviews"].isna().any()

100%|██████████| 70/70 [04:15<00:00,  3.66s/it]


In [143]:
users = user[user["number_reviews"]>=MIN_SHOWS]
print(user.shape[0])
users=users.sample(frac=DATA_TO_TAKE,random_state=SEEDS[0])
print(users.shape)

1123284
(115892, 3)


In [144]:
print(user.columns)
user.head()
users[users.number_reviews<50]

Index(['num_completed', 'mean_score', 'number_reviews'], dtype='object')


,num_completed,mean_score,number_reviews
user_id,,,


In [145]:

train_users = users.sample(frac=TRAIN_TEST_SPLITS[0],random_state=SEEDS[1])
train_users = pd.DataFrame(index = train_users.index,
                           data=np.arange(train_users.shape[0]))
validation_users = users.drop(train_users.index)
validation_users = pd.DataFrame(index = validation_users.index,
                           data=np.arange(validation_users.shape[0]))
train_users.to_csv(OUTPUT_TRAIN_USERS,index_label="user_id")
validation_users.to_csv(OUTPUT_VALIDATION_USERS,index_label="user_id")
del validation_users

In [146]:
def normalize(x):
    t = x!=0
    means = x.sum(axis=-1)/t.sum(axis=-1)
    x[:]  = np.where(t,x - means[:,np.newaxis],0)
    x[:]/=np.linalg.norm(x,axis=-1)[:,np.newaxis]
    x[:] = np.nan_to_num(x)

In [147]:
train_vals = np.zeros((train_users.shape[0],top_animes.shape[0]),dtype=np.float32)

In [148]:
NUM_USER_ANIME_FILE=70
cols_to_use = ['anime_id', 'user_id', 'score', 'favorite']
for x in trange(NUM_USER_ANIME_FILE):
    file_name = "{}/user_anime{:012d}.csv".format(svanoo_myanimelist_dataset_path,x)
    df = pd.read_csv(
        file_name,
        sep='\t',
        keep_default_na=False,
        na_values=("-------",""),
        low_memory=False,
        usecols=cols_to_use  # Only read needed columns
    ).dropna()
    df = df[df["anime_id"].isin(top_animes.index)]
    df= df[df["user_id"].isin(train_users.index)]
    us = df["user_id"].values
    user_numbers = train_users.loc[us,0].values
    anime_numbers = top_animes.loc[df["anime_id"].values,"number"]
    train_vals[user_numbers,anime_numbers] = df["score"].values

100%|██████████| 70/70 [03:53<00:00,  3.34s/it]


In [149]:
BATCH_SIZE = 256
size=train_vals.shape[0]
print(np.sum(np.sum(train_vals!=0,axis=1)==0))
for start in trange(0,size,BATCH_SIZE):
    end = min(start+BATCH_SIZE,size)
    normalize(train_vals[start:end,:])
np.save(OUTPUT_TRAINING_VALUES,train_vals)

0


  0%|          | 0/408 [00:00<?, ?it/s]C:\Users\Heart\AppData\Local\Temp\ipykernel_4392\249866724.py:5: RuntimeWarning: invalid value encountered in divide
  x[:]/=np.linalg.norm(x,axis=-1)[:,np.newaxis]
100%|██████████| 408/408 [00:03<00:00, 106.83it/s]


In [150]:
print(train_vals[:10,:10])

[[ 0.1202057   0.07623785 -0.01169787  0.03226999  0.          0.03226999
  -0.01169787  0.07623785  0.03226999  0.        ]
 [ 0.          0.0915111  -0.11986665  0.0915111  -0.11986665  0.0915111
   0.0915111  -0.43693325  0.          0.0915111 ]
 [ 0.         -0.0338255  -0.16912752  0.          0.          0.
   0.         -0.0338255   0.         -0.0338255 ]
 [-0.00825151  0.07013784 -0.16503021  0.          0.07013784  0.
   0.          0.07013784  0.          0.        ]
 [ 0.03575903  0.03575903  0.          0.03575903  0.03575903  0.03575903
   0.03575903  0.03575903  0.03575903  0.03575903]
 [ 0.          0.          0.          0.08702686  0.         -0.0607546
   0.08702686  0.         -0.0607546   0.        ]
 [-0.02952841  0.01682531  0.          0.06317903  0.10953275  0.01682531
   0.06317903  0.01682531  0.01682531  0.10953275]
 [-0.05544735  0.03853121 -0.05544735  0.08552051  0.08552051  0.03853121
   0.          0.          0.03853121  0.        ]
 [ 0.          0.1

In [151]:
validation_users = pd.read_csv(OUTPUT_VALIDATION_USERS,index_col="user_id")
validation_values = np.zeros((validation_users.shape[0],top_animes.shape[0]),dtype=np.float32)
NUM_USER_ANIME_FILE=70
cols_to_use = ['anime_id', 'user_id', 'score', 'favorite']
for x in trange(NUM_USER_ANIME_FILE):
    file_name = "{}/user_anime{:012d}.csv".format(svanoo_myanimelist_dataset_path,x)
    df = pd.read_csv(
        file_name,
        sep='\t',
        keep_default_na=False,
        na_values=("-------",""),
        low_memory=False,
        usecols=cols_to_use  # Only read needed columns
    ).dropna()
    df = df[df["anime_id"].isin(top_animes.index)]
    df= df[df["user_id"].isin(validation_users.index)]
    us = df["user_id"].values
    user_numbers = validation_users.loc[us,"number"].values
    anime_numbers = top_animes.loc[df["anime_id"].values,"number"]
    validation_values[user_numbers,anime_numbers] = df["score"].values
normalize(validation_values)

  0%|          | 0/70 [00:04<?, ?it/s]


KeyError: 'number'

In [ ]:
print(validation_values[0])

In [ ]:
train_anime=top_animes.sample(frac=TRAIN_TEST_SPLITS[1],random_state=SEEDS[2])
train_anime_bool = top_animes.index.isin(train_anime.index)
test_anime = top_animes[~train_anime_bool].copy()
print(test_anime.shape)

In [ ]:
class MaximumStructure:
    # INITIAL = 200
    def __init__(self,keys):
        keep=keys>0
        self.inds = np.arange(keys.shape[0])
        self.inds= self.inds[keep]
        self.keys = keys[keep]
        self.shape = self.keys.shape[0]
        self.split = self.shape
        self.heapify()

    def swap(self,i,j):
        self.keys[i],self.keys[j] = self.keys[j],self.keys[i]
        self.inds[j],self.inds[i]=self.inds[i],self.inds[j]

    def sift_down(self,i):
        val_cur = self.keys[i]
        if 2*i+1<self.split:
            left_cur = self.keys[2*i+1]
        else:
            left_cur=-np.inf
        right_cur= self.keys[2*i+2] if 2*i+2<self.split else -np.inf
        if val_cur<right_cur and left_cur<=right_cur:
            selected=2*i+2
        elif val_cur<left_cur:
            selected=2*i+1
        else:
            return
        self.swap(i,selected)
        self.sift_down(selected)

    def heapify(self):
        # i = self.keys.argpartition(MaximumStructure.INITIAL)
        # self.keys=self.keys[i]
        # self.inds=self.inds[i]
        # i=np.argsort(self.keys[-MaximumStructure.INITIAL:])
        # self.keys[- MaximumStructure.INITIAL:]=self.keys[i]
        # self.inds[- MaximumStructure.INITIAL:]=self.inds[i]
        # self.split = self.shape - MaximumStructure.INITIAL
        for j in range((self.split-1)//2,-1,-1):
            self.sift_down(j)


    def extract_max(self):
        self.split-=1
        self.swap(0,self.split)
        self.sift_down(0)
        return self.inds[self.split],self.keys[self.split]

    def extracted(self):
        return self.inds[self.split:],self.keys[self.split:]

    def not_extracted(self):
        return self.split!=0

    def __iter__(self):
        for i in range(self.shape-1,self.split-1,-1):
            yield self.inds[i],self.keys[i]
        while self.split>-self.shape:
            yield self.extract_max()

In [ ]:
train_vals = np.load("training_vals.npy")

In [ ]:
squared_error = np.zeros(KS_TO_TRY)
MASK = np.arange(KS_TO_TRY)[:,np.newaxis]>=np.arange(KS_TO_TRY)
size=validation_values.shape[0]
weighted_square_error = 0
arith_square_error = 0
samples_taken = 0
amm_sum=0
for i in trange(size//6):
    test_cur = validation_values[i]
    test_cur_to_search = np.where(train_anime_bool,test_cur,0)
    similarities = np.inner(train_vals,test_cur_to_search)
    similarities = MaximumStructure(similarities)
    for ind,row in test_anime.iterrows():
        anime_number = row["number"]
        act_rate = test_cur[anime_number]
        if act_rate==0:
            continue
        # t = act_rate!=0
        # anime = anime[t]
        # act_rate=act_rate[t]
        # ratings = train_vals[:,anime]
        # bool_anime  = ratings == 0
        # altered_similiarity = np.where(bool_anime,0,similarities[:,np.newaxis])
        # samples_taken+=anime.shape[0]
        # indices = np.argpartition(altered_similiarity,similarities.shape[0] -AMOUNT, axis=0)[-AMOUNT:]
        # sims = altered_similiarity[indices.T]
        # rates = ratings[indices.T]
        extracted_inds,extracted_similarities = similarities.extracted()
        t = train_vals[extracted_inds,anime_number]!=0
        extracted_inds,extracted_similarities = extracted_inds[t],extracted_similarities[t]
        amm = min(KS_TO_TRY,extracted_inds.shape[0])
        sims= np.zeros(KS_TO_TRY)
        rates = np.ndarray(KS_TO_TRY)
        rates[:amm] = train_vals[extracted_inds[:amm],anime_number]
        sims[:amm] = extracted_similarities[:amm]
        while amm<KS_TO_TRY and similarities.not_extracted():
            i,k = similarities.extract_max()
            rat = train_vals[i,anime_number]
            if rat!=0:
                sims[amm]=k
                rates[amm]=rat
                amm+=1
        amm_sum+=amm
        if amm ==0:
            continue
        new_rates = np.where(MASK,rates[:,np.newaxis],0)
        new_sims = np.where(MASK,sims[:,np.newaxis],0)
        samples_taken+=1
        weighted_rating = np.average(new_rates,weights=new_sims,axis=0)
        weighted_square_error += (weighted_rating - act_rate)**2
        new_mask = np.where(MASK[amm-1],MASK,0)
        arith_rating  = np.average(new_rates,weights=new_mask,axis=0)
        arith_square_error += (arith_rating - act_rate)**2

In [ ]:
from matplotlib import pyplot as plt
plt.plot(np.arange(1,13),weighted_square_error[:12]/samples_taken,label="Ratings taken with weighed average")
plt.plot(np.arange(1,13),arith_square_error[:12]/samples_taken,label="Ratings taken with arithmetic average")
plt.xlabel("Amount of Users taken")
plt.ylabel("Mean squared error")
plt.title("Error of Recommendation System by the amount of titles compared to")
plt.legend(loc="best")
plt.show()

In [ ]:
print(amm_sum/samples_taken)

In [ ]:
print((arith_square_error-weighted_square_error)/samples_taken)

In [ ]:
train_vals = np.load("training_vals.npy")

In [ ]:
squared_error = np.zeros(KS_TO_TRY)
MASK = np.arange(KS_TO_TRY)[:,np.newaxis]>=np.arange(KS_TO_TRY)
size=validation_values.shape[0]
weighted_inner_product = np.zeros(KS_TO_TRY)
arith_inner_product = np.zeros(KS_TO_TRY)
weighted_sq_norm = np.zeros(KS_TO_TRY)
arith_sq_norm = np.zeros(KS_TO_TRY)
samples_taken = 0
amm_sum=0
for i in trange(size//6):
    test_cur = validation_values[i]
    test_cur_to_search = np.where(train_anime_bool,test_cur,0)
    similarities = np.inner(train_vals,test_cur_to_search)
    similarities = MaximumStructure(similarities)
    weighted_ratings = np.zeros((KS_TO_TRY,top_animes.shape[0]))
    arith_ratings = np.zeros((KS_TO_TRY,top_animes.shape[0]))
    for ind,row in test_anime.iterrows():
        anime_number = row["number"]
        act_rate = test_cur[anime_number]
        if act_rate==0:
            continue
        # t = act_rate!=0
        # anime = anime[t]
        # act_rate=act_rate[t]
        # ratings = train_vals[:,anime]
        # bool_anime  = ratings == 0
        # altered_similiarity = np.where(bool_anime,0,similarities[:,np.newaxis])
        # samples_taken+=anime.shape[0]
        # indices = np.argpartition(altered_similiarity,similarities.shape[0] -AMOUNT, axis=0)[-AMOUNT:]
        # sims = altered_similiarity[indices.T]
        # rates = ratings[indices.T]
        extracted_inds,extracted_similarities = similarities.extracted()
        t = train_vals[extracted_inds,anime_number]!=0
        extracted_inds,extracted_similarities = extracted_inds[t],extracted_similarities[t]
        amm = min(KS_TO_TRY,extracted_inds.shape[0])
        sims= np.zeros(KS_TO_TRY)
        rates = np.ndarray(KS_TO_TRY)
        rates[:amm] = train_vals[extracted_inds[:amm],anime_number]
        sims[:amm] = extracted_similarities[:amm]
        while amm<KS_TO_TRY and similarities.not_extracted():
            i,k = similarities.extract_max()
            rat = train_vals[i,anime_number]
            if rat!=0:
                sims[amm]=k
                rates[amm]=rat
                amm+=1
        amm_sum+=amm
        if amm ==0:
            continue
        new_rates = np.where(MASK,rates[:,np.newaxis],0)
        new_sims = np.where(MASK,sims[:,np.newaxis],0)
        samples_taken+=1
        weighted_ratings[:,anime_number] = np.average(new_rates,weights=new_sims,axis=0)
        new_mask = np.where(MASK[amm-1],MASK,0)
        arith_ratings[:,anime_number] = np.average(new_rates,weights=new_mask,axis=0)
    weighted_inner_product+=np.inner(weighted_ratings,test_cur)
    arith_inner_product += np.inner(arith_ratings,test_cur)
    weighted_sq_norm += np.sum(weighted_ratings ** 2,axis=1)
    arith_sq_norm += np.sum(arith_ratings ** 2,axis=1)

In [ ]:
validation_users_test_anime = validation_values[:size//6,~train_anime_bool]
norm = np.linalg.norm(validation_users_test_anime,ord="fro")
weighted_cosine = weighted_inner_product/(np.sqrt(weighted_sq_norm)*norm)
arith_cosine = arith_inner_product/(np.sqrt(arith_sq_norm)*norm)

In [ ]:
print(weighted_inner_product)
print(arith_inner_product)
print(norm)
print(validation_users_test_anime)

In [ ]:
from matplotlib import pyplot as plt
plt.plot(np.arange(1,KS_TO_TRY+1),weighted_cosine,label="Ratings taken with weighed average")
plt.plot(np.arange(1,KS_TO_TRY+1),arith_cosine,label="Ratings taken with arithmetic average")
plt.xlabel("Amount of Users taken")
plt.ylabel("Mean squared error")
plt.title("Error of Recommendation System by the amount of titles compared to")
plt.legend(loc="best")
plt.show()
plt.close()

In [ ]:
squared_error = np.zeros(KS_TO_TRY)
MASK = np.arange(KS_TO_TRY)[:,np.newaxis]>=np.arange(KS_TO_TRY)
size=validation_values.shape[0]
weighted_inner_product = np.zeros(KS_TO_TRY)
arith_inner_product = np.zeros(KS_TO_TRY)
weighted_sq_norm = np.zeros(KS_TO_TRY)
arith_sq_norm = np.zeros(KS_TO_TRY)
samples_taken = 0
amm_sum=0
for i in trange(size//6):
    test_cur = validation_values[i]
    test_cur_to_search = np.where(train_anime_bool,test_cur,0)

    t=test_cur_to_search!=0
    cur_train_vals = train_vals[:,t]
    similarities = np.inner(cur_train_vals,test_cur_to_search[t])/np.linalg.norm(cur_train_vals,axis=1)
    similarities = MaximumStructure(similarities)
    weighted_ratings = np.zeros((KS_TO_TRY,top_animes.shape[0]))
    arith_ratings = np.zeros((KS_TO_TRY,top_animes.shape[0]))
    for ind,row in test_anime.iterrows():
        anime_number = row["number"]
        act_rate = test_cur[anime_number]
        if act_rate==0:
            continue
        # t = act_rate!=0
        # anime = anime[t]
        # act_rate=act_rate[t]
        # ratings = train_vals[:,anime]
        # bool_anime  = ratings == 0
        # altered_similiarity = np.where(bool_anime,0,similarities[:,np.newaxis])
        # samples_taken+=anime.shape[0]
        # indices = np.argpartition(altered_similiarity,similarities.shape[0] -AMOUNT, axis=0)[-AMOUNT:]
        # sims = altered_similiarity[indices.T]
        # rates = ratings[indices.T]
        extracted_inds,extracted_similarities = similarities.extracted()
        t = train_vals[extracted_inds,anime_number]!=0
        extracted_inds,extracted_similarities = extracted_inds[t],extracted_similarities[t]
        amm = min(KS_TO_TRY,extracted_inds.shape[0])
        sims= np.zeros(KS_TO_TRY)
        rates = np.ndarray(KS_TO_TRY)
        rates[:amm] = train_vals[extracted_inds[:amm],anime_number]
        sims[:amm] = extracted_similarities[:amm]
        while amm<KS_TO_TRY and similarities.not_extracted():
            i,k = similarities.extract_max()
            rat = train_vals[i,anime_number]
            if rat!=0:
                sims[amm]=k
                rates[amm]=rat
                amm+=1
        amm_sum+=amm
        if amm ==0:
            continue
        new_rates = np.where(MASK,rates[:,np.newaxis],0)
        new_sims = np.where(MASK,sims[:,np.newaxis],0)
        samples_taken+=1
        weighted_ratings[:,anime_number] = np.average(new_rates,weights=new_sims,axis=0)
        new_mask = np.where(MASK[amm-1],MASK,0)
        arith_ratings[:,anime_number] = np.average(new_rates,weights=new_mask,axis=0)
    weighted_inner_product+=np.inner(weighted_ratings,test_cur)
    arith_inner_product += np.inner(arith_ratings,test_cur)
    weighted_sq_norm += np.sum(weighted_ratings ** 2,axis=1)
    arith_sq_norm += np.sum(arith_ratings ** 2,axis=1)

In [ ]:
validation_users_test_anime = validation_values[:size//6,~train_anime_bool]
norm = np.linalg.norm(validation_users_test_anime,ord="fro")
weighted_cosine = weighted_inner_product/(np.sqrt(weighted_sq_norm)*norm)
arith_cosine = arith_inner_product/(np.sqrt(arith_sq_norm)*norm)

In [ ]:
print(arith_cosine,weighted_cosine)

In [ ]:
from matplotlib import pyplot as plt
plt.plot(np.arange(1,KS_TO_TRY+1),weighted_cosine,label="Ratings taken with weighed average")
plt.plot(np.arange(1,16),arith_cosine,label="Ratings taken with arithmetic average")
plt.xlabel("Amount of Users taken")
plt.ylabel("Cosine Similarity")
plt.title("Error of Recommendation System by the amount of titles compared to")
plt.legend(loc="best")
plt.show()